# BhashaCore — where does a multilingual chatbot lose quality?

**SIH25104 · Language-Agnostic Chatbot.** Government chatbots answer worse in regional Indian languages. This notebook builds a small chatbot pipeline and finds the exact stage where each language loses accuracy.

The test is controlled. Every language gets **the same 900 questions** (Belebele), with the same correct option. So a lower score comes from the language, not from harder questions.

```
 user question (any language, any script)
        │
 [1] Language ID ............ GlotLID                              → scored: label accuracy
        │
 [2] Retrieve the passage ... multilingual-e5 (dense)              → scored: Recall@1
        │
   ┌────┴─────────────────────────┐
 [3A] Direct                    [3B] Pivot
  LLM reads the original text    NLLB translates to English         → scored: chrF
                                 then the same LLM reads English
   └────┬─────────────────────────┘
   option A / B / C / D ...................................... → scored: accuracy, gap vs English
```

| Section | What happens |
|---|---|
| 0–1 | Install, settings |
| 2 | Load Belebele (scoring data) and check that languages are paired row by row |
| 3 | Load a random sample of IndicCorp v2 (raw text, for the token tax) |
| 4 | **Stage 1** — language identification with GlotLID |
| 5 | Token tax — how many tokens each language costs |
| 6 | **Stage 2** — retrieval with multilingual-e5 embeddings |
| 7–8 | **Stage 3A** — the LLM answers from the original-language passage |
| 9–10 | **Stage 3B** — translate to English with NLLB, then answer |
| 11 | End to end — all stages chained, errors pass from stage to stage |
| 12–14 | Comparison report, paired statistics, failure examples |
| 15 | Write the report files |
| 16 | Chatbot demo with free-text answers |

**Hardware.** The defaults run on an **8 GB laptop GPU with 16 GB of RAM** (tested on an RTX 4060 Laptop under WSL). The LLM is loaded with 4-bit weights, and on a small GPU only one model sits on the GPU at a time (Section 1 explains why). On a GPU with 20 GB or more, set `LLM_4BIT = False`. The first run downloads about 16 GB of models. Every expensive result is saved in `bhashacore_outputs/cache/`, so a second run is fast.

## 0. Install

Install PyTorch 2.6 or newer with CUDA first, from [pytorch.org](https://pytorch.org/get-started/locally/). Version 2.6 is the minimum that current transformers versions support. Do not install `torch` with the line below, because pip can replace your CUDA build with a CPU build. Then run this cell once.

In [ ]:
%pip install -q --root-user-action=ignore --disable-pip-version-check "transformers>=4.56" "sentence-transformers>=3.0" "pandas>=2.1" datasets huggingface_hub accelerate bitsandbytes sentencepiece sacrebleu scipy statsmodels tabulate fasttext matplotlib tqdm

In [ ]:
import ctypes, gc, os, re, json, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
from datasets import load_dataset
from huggingface_hub import HfFileSystem, hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import fasttext
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import mcnemar
import sacrebleu

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 140)

# Quiet library notices (errors still show)
from huggingface_hub.utils import logging as hf_hub_logging
hf_hub_logging.set_verbosity_error()                  # hide the Hub's "set a HF_TOKEN" notice
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(torch.cuda.get_device_name(0), f"| {total / 1e9:.1f} GB, {free / 1e9:.1f} GB free")

## 1. Settings

Change the values in the next cell, then run all cells.

- **Languages.** Codes are Belebele / FLORES-200 codes: `<language>_<script>`. For example, `hin_Deva` is Hindi in the Devanagari script. The first entry is the baseline that every other language is compared with.
- **Models.** One model per stage, all from Hugging Face. None of them needs an access request.
- **`N_QUESTIONS`.** Use `200` for a fast first run. Use `None` for all 900 questions in the final report.

**Memory on a small GPU.** The retriever, the LLM and the translator need about 7 GB of GPU memory together, too much for an 8 GB laptop GPU, and parking the idle ones in RAM would need 6 GB of a 16 GB laptop's RAM. So when the GPU has less than 20 GB, **only one of them is loaded at a time**: before a model works, the others are removed from memory, and they are loaded again from disk when they are needed (5–10 seconds each). The notebook runs each stage for all languages before the next stage, so this happens only a few times. The notebook also caps PyTorch's GPU memory below what is free. On Windows and WSL this matters: the NVIDIA driver does not stop at the card's memory, it quietly borrows system RAM, which can freeze or crash the computer. With the cap you get a clear `CUDA out of memory` error instead. If you see it, lower the batch sizes.

In [ ]:
# ---------- what to test ----------
EVAL_LANGS = ["eng_Latn", "hin_Deva", "mar_Deva", "ben_Beng", "tam_Taml", "tel_Telu"]
BASELINE = "eng_Latn"

LANG_NAMES = {
    "eng_Latn": "English", "hin_Deva": "Hindi", "mar_Deva": "Marathi", "ben_Beng": "Bengali",
    "tam_Taml": "Tamil", "tel_Telu": "Telugu", "asm_Beng": "Assamese", "guj_Gujr": "Gujarati",
    "kan_Knda": "Kannada", "mal_Mlym": "Malayalam", "ory_Orya": "Odia", "pan_Guru": "Punjabi", "urd_Arab": "Urdu",
}

# IndicCorp v2 file code -> Belebele code, for the token-tax measurement (Section 5).
# Every language in EVAL_LANGS needs an entry. Other IndicCorp codes:
# as Assamese, gu Gujarati, kn Kannada, ml Malayalam, or Odia, pa Punjabi, ur Urdu, ne Nepali, sa Sanskrit.
INDICCORP_LANGS = {"en": "eng_Latn", "hi": "hin_Deva", "mr": "mar_Deva", "bn": "ben_Beng", "ta": "tam_Taml", "te": "tel_Telu"}

# ---------- models (one per stage) ----------
LID_REPO, LID_FILE = "cis-lmu/glotlid", "model_v4.bin"     # Stage 1: GlotLID v4 (a pinned file, so results do not move)
EMB_NAME = "intfloat/multilingual-e5-large-instruct"      # Stage 2: sentence embeddings for retrieval
EMB_QUERY_PREFIX = "Instruct: Given a question, retrieve the passage that answers it\nQuery: "
LLM_NAME = "Qwen/Qwen3-4B-Instruct-2507"                  # Stage 3: answering model
LLM_4BIT = True                                           # 4-bit weights: 2.6 GB instead of 8 GB. GPU with 20 GB or more: False
MT_NAME = "facebook/nllb-200-distilled-1.3B"              # Stage 3B: translator for the pivot route
# The main branch of NLLB has only a float32 .bin file: loading it needs 4.5 GB of RAM at once. This revision is
# Hugging Face's own safetensors conversion of the same weights (PR #5), which loads straight into bfloat16.
MT_REVISION = "f9e6ff7fc090740fcce5af7d7636cb27a33d0f18"

# ---------- size and speed ----------
N_QUESTIONS = None          # None = all 900 questions; 200 = quick first run
INDICCORP_LINES = 6000      # sentences kept per language
INDICCORP_CHUNKS = 40       # random places read in each IndicCorp file
INDICCORP_CHUNK_BYTES = 256 * 1024
INDICCORP_LOCAL_DIR = None  # optional: a folder with <code>.txt files (e.g. "data/indiccorp") if the download fails
LLM_BATCH, MT_BATCH, EMB_BATCH = 4, 16, 32   # 8 GB GPU; lower them if you see "CUDA out of memory", raise them on a big GPU
MT_BEAMS, MT_MAX_TOKENS = 4, 256             # beam search: better translations than greedy (1), but slower
GPU_HEADROOM_GB = 0.8       # GPU memory kept free for the display and the CUDA driver
CHRF_DAMAGED = 40.0         # a passage translation below this chrF counts as "damaged"
N_FAILURE_EXAMPLES = 5      # failure examples per language in the report
SEED = 42
USE_CACHE = True            # reuse saved results; set False to recompute everything
OUT_DIR = Path("bhashacore_outputs")

In [ ]:
CACHE_DIR, FIG_DIR = OUT_DIR / "cache", OUT_DIR / "figures"
for d in (OUT_DIR, CACHE_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif torch.backends.mps.is_available():
    DEVICE, DTYPE = "mps", torch.float16
else:
    DEVICE, DTYPE = "cpu", torch.float32
MT_DTYPE = torch.float32 if DTYPE == torch.float16 else DTYPE   # NLLB can overflow in float16
QTAG = f"n{N_QUESTIONS or 'all'}"                                  # part of every cache file name
LLM_4BIT = LLM_4BIT and DEVICE == "cuda"                           # bitsandbytes 4-bit needs a CUDA GPU

# ---------- GPU memory ----------
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == "cuda" else 0.0
ONE_MODEL_ON_GPU = DEVICE == "cuda" and VRAM_GB < 20
if DEVICE == "cuda":
    # Cap PyTorch below the free memory: a clear "CUDA out of memory" error instead of the driver borrowing system RAM
    free, total = torch.cuda.mem_get_info()
    torch.cuda.set_per_process_memory_fraction(max(0.2, (free - GPU_HEADROOM_GB * 1e9) / total))

MODELS = {}     # name -> loaded model
LOADERS = {}    # name -> function that loads the model onto DEVICE

def free_memory():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)        # Linux: hand freed RAM back to the system
    except (OSError, AttributeError):
        pass

def gpu(name):
    # Return model `name`, loaded on DEVICE. With ONE_MODEL_ON_GPU, first remove every other model from memory;
    # LOADERS loads it again from disk when it is needed. Keep no other reference to a model, or it cannot be freed.
    if ONE_MODEL_ON_GPU and any(n != name for n in MODELS):
        for other in [n for n in MODELS if n != name]:
            del MODELS[other]
        free_memory()
    if name not in MODELS:
        MODELS[name] = LOADERS[name]()
    return MODELS[name]

import transformers
print("device:", DEVICE, "| dtype:", DTYPE, "| transformers", transformers.__version__,
      f"| GPU {VRAM_GB:.0f} GB, one model on the GPU at a time: {ONE_MODEL_ON_GPU}")


def name(code):
    return LANG_NAMES.get(code, code)

missing = [l for l in EVAL_LANGS if l not in INDICCORP_LANGS.values()]
assert not missing, f"Add these languages to INDICCORP_LANGS (IndicCorp code -> Belebele code): {missing}"

def slug(text):
    return re.sub(r"[^A-Za-z0-9]+", "-", str(text)).strip("-")

def chunks(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]

def cached(key, compute):
    # Run compute() once and save its result. Later runs load the saved file.
    path = CACHE_DIR / f"{slug(key)}.pkl"
    if USE_CACHE and path.exists():
        return pd.read_pickle(path)
    result = compute()
    pd.to_pickle(result, path)
    return result

def save_fig(fig, filename):
    fig.savefig(FIG_DIR / filename, dpi=150, bbox_inches="tight")
    plt.show()

def md_table(df, digits=1):
    # DataFrame -> Markdown table text, for the report file. Floats are rounded and NaN shows as "–";
    # disable_numparse keeps text such as the p-value "8.1e-60" exactly as it is.
    shown = df.map(lambda v: ("–" if np.isnan(v) else f"{v:.{digits}f}") if isinstance(v, (float, np.floating)) else v)
    return shown.to_markdown(disable_numparse=True)

## 2. Belebele — the scoring data

Belebele has 900 reading questions. Each question has a short news passage, four options and one correct option. Experts translated every passage, question and option into 122 language variants. Thus row *i* in Tamil is the translation of row *i* in English.

The next cell does three things:

1. It downloads one table per language.
2. It checks that `link` and `question_number` match in every row. If they do not match, it reorders the rows by these two keys, or it stops with an error.
3. It gives every row a question id (`qid`) and every distinct passage a passage id (`pid`). These ids are the same in every language.

The **knowledge base** `KB[lang]` holds one row per passage. The retriever searches it in Section 6.

In [ ]:
def load_belebele(lang):
    df = load_dataset("facebook/belebele", lang, split="test").to_pandas()
    df["answer"] = df["correct_answer_num"].astype(int) - 1          # "1".."4"  ->  0..3
    df["options"] = df[["mc_answer1", "mc_answer2", "mc_answer3", "mc_answer4"]].values.tolist()
    df["key"] = df["link"] + "#" + df["question_number"].astype(str)
    df["key"] = df["key"] + "#" + df.groupby("key").cumcount().astype(str)   # unique even if a link repeats
    return df[["key", "flores_passage", "question", "options", "answer"]]

raw = {lang: load_belebele(lang) for lang in tqdm(EVAL_LANGS, desc="Belebele")}

# Pair the rows: row i must be the same question in every language
ref_keys = raw[BASELINE]["key"].to_numpy()
for lang, df in raw.items():
    if not np.array_equal(df["key"].to_numpy(), ref_keys):
        if df["key"].is_unique and raw[BASELINE]["key"].is_unique:
            raw[lang] = df.set_index("key").loc[ref_keys].reset_index()
            print(f"{lang}: rows reordered to match {BASELINE}")
        else:
            raise ValueError(f"{lang}: cannot pair rows with {BASELINE}")
    same_answers = (raw[lang]["answer"].to_numpy() == raw[BASELINE]["answer"].to_numpy()).mean()
    print(f"{name(lang):20s} rows={len(raw[lang])}  same correct option as {BASELINE}: {same_answers:.0%}")

# Ids shared by all languages
pid_of_row = pd.factorize(raw[BASELINE]["flores_passage"])[0]
all_qids = np.arange(len(raw[BASELINE]))
qids = all_qids if N_QUESTIONS is None else np.sort(
    np.random.default_rng(SEED).choice(all_qids, N_QUESTIONS, replace=False))

B, KB = {}, {}
for lang, df in raw.items():
    df = df.assign(qid=all_qids, pid=pid_of_row)
    KB[lang] = df.drop_duplicates("pid").set_index("pid")["flores_passage"]   # all passages, even in a quick run
    B[lang] = df.set_index("qid").loc[qids]

print(f"\nquestions per language: {len(qids)} | passages in each knowledge base: {len(KB[BASELINE])}")

In [ ]:
# One question, in every language. The passage is cut to 160 characters for display.
q = qids[0]
pd.DataFrame([{
    "language": name(lang),
    "passage (start)": B[lang].loc[q, "flores_passage"][:160] + " ...",
    "question": B[lang].loc[q, "question"],
    "correct option": B[lang].loc[q, "options"][B[lang].loc[q, "answer"]],
} for lang in EVAL_LANGS]).set_index("language")

## 3. IndicCorp v2 — raw text for each language

IndicCorp v2 (AI4Bharat) holds billions of sentences from Indian news and web pages, one text file per language. It has **no questions and no answers**, so it cannot score a chatbot. We use it to measure how many tokens each language needs (Section 5), on ordinary text from a different source than Belebele.

The files are large: 1 GB to 80 GB each. We do not download them. `huggingface_hub.HfFileSystem` opens a Hub file like a local file: `seek()` and `read()` become HTTP *range requests*. We read 40 small pieces (256 KB each) at random places and keep complete sentences only. Random places give a fairer mix of topics than the first lines of a file, which come from a few websites only.

If the download fails, download any part of a file by hand from the [dataset page](https://huggingface.co/datasets/ai4bharat/IndicCorpV2), save it as `<folder>/<code>.txt` (for example `data/indiccorp/ta.txt`), and set `INDICCORP_LOCAL_DIR` in Section 1.

In [ ]:
HF_FS = HfFileSystem()     # uses HF_ENDPOINT and HF_TOKEN from the environment if they are set

def open_indiccorp(code):
    # Return (file object, size in bytes). A local copy if there is one, otherwise the file on the Hub.
    # Both support seek() and read(), so the sampling code below does not care which one it gets.
    local = Path(INDICCORP_LOCAL_DIR) / f"{code}.txt" if INDICCORP_LOCAL_DIR else None
    if local is not None and local.exists():
        return open(local, "rb"), local.stat().st_size
    path = f"datasets/ai4bharat/IndicCorpV2/data/{'hi-1.txt' if code == 'hi' else code + '.txt'}"   # Hindi is split into 3 files
    # block_size = one piece, so each read() downloads about one piece and not more
    return HF_FS.open(path, "rb", block_size=INDICCORP_CHUNK_BYTES), HF_FS.size(path)

def sample_indiccorp(code):
    cache = CACHE_DIR / f"indiccorp_{code}_{INDICCORP_LINES}.txt"
    if cache.exists():
        return cache.read_text(encoding="utf-8").split("\n")
    rng = random.Random(f"{SEED}-{code}")
    f, size = open_indiccorp(code)
    lines = []
    with f:
        for _ in range(INDICCORP_CHUNKS):         # pieces at random offsets; keep only the complete lines
            start = rng.randrange(0, max(1, size - INDICCORP_CHUNK_BYTES))
            f.seek(start)
            parts = f.read(INDICCORP_CHUNK_BYTES).decode("utf-8", errors="ignore").split("\n")
            lines += parts[1:-1] if start > 0 else parts[:-1]    # first and last parts are cut lines
    seen, kept = set(), []
    for line in lines:
        line = line.strip()
        if len(line.split()) >= 4 and line not in seen:   # drop headings, very short lines and repeats
            seen.add(line)
            kept.append(line)
    rng.shuffle(kept)
    kept = kept[:INDICCORP_LINES]
    cache.write_text("\n".join(kept), encoding="utf-8")
    return kept

corpus = {code: sample_indiccorp(code) for code in tqdm(INDICCORP_LANGS, desc="IndicCorp")}
pd.DataFrame([{
    "IndicCorp code": code, "language": name(INDICCORP_LANGS[code]), "sentences": len(lines),
    "avg words per sentence": np.mean([len(l.split()) for l in lines]), "example": lines[0][:90],
} for code, lines in corpus.items()]).set_index("IndicCorp code").round(1)

## 4. Stage 1 — Language identification

**Job.** Read the user's message and name its language. Later stages need this label: the bot searches the knowledge base of that language, and the translator needs the source language.

**Model — [GlotLID](https://huggingface.co/cis-lmu/glotlid) v4** (a fastText classifier). It works in three steps:

1. **Character n-grams.** The text is cut into pieces of 2 to 5 characters inside each word. For example, `नमस्ते` gives `नम`, `नमस`, `मस`, ... Each piece has a learned vector of 96 numbers.
2. **Average.** The vectors of all pieces and words are averaged into one vector for the whole message.
3. **Softmax** over **2060 language-script labels**: one learned weight vector per label, and the label with the highest score wins.

It was trained on web text in all 2060 labels, including short random spans of text, so it also works on chat-length messages. We keep only the labels this bot serves (`EVAL_LANGS`) and pick the most probable of those: a message is never sent to a language the bot has no knowledge base for.

**Why characters, not words?** Character pieces work for words the model never saw, and for spelling variants. Script alone is not enough: Hindi and Marathi both use Devanagari, so the model must learn the words.

**Test.** Belebele *questions*. These are short and come from a different source than the training text, like real chat messages.

In [ ]:
glotlid = fasttext.load_model(hf_hub_download(LID_REPO, LID_FILE))     # runs on the CPU, ~1 GB of RAM
LID_TAG = {f"__label__{code}": code for code in EVAL_LANGS}             # the languages this bot serves
print(f"GlotLID knows {len(glotlid.get_labels())} labels; we keep {len(LID_TAG)}:", ", ".join(EVAL_LANGS))

def detect_language(texts):
    # k=-1 returns every label, most probable first. The first label we serve is the best of them.
    # (We pass a list: fastText's single-text predict() fails on NumPy 2, the list version works.)
    labels, _ = glotlid.predict([t.replace("\n", " ") for t in texts], k=-1)
    return np.array([next(LID_TAG[l] for l in row if l in LID_TAG) for row in labels])

detect_language(["विजेचे बिल कुठे भरायचे?", "बिजली का बिल कहाँ जमा करें?", "Where do I pay my electricity bill?"])

In [ ]:
# Test on Belebele questions (short texts from a different source)
LID_PRED = {lang: pd.Series(detect_language(B[lang]["question"].tolist()), index=B[lang].index) for lang in EVAL_LANGS}
lid_acc = {lang: 100 * (LID_PRED[lang] == lang).mean() for lang in EVAL_LANGS}

# Shorter messages are harder: keep only the first k words of each question
short = {}
for k in (1, 2, 3, 5):
    short[f"first {k} word{'s' if k > 1 else ''}"] = {
        name(lang): 100 * (detect_language([" ".join(q.split()[:k]) for q in B[lang]["question"]]) == lang).mean()
        for lang in EVAL_LANGS}
short["full question"] = {name(lang): lid_acc[lang] for lang in EVAL_LANGS}
lid_short_df = pd.DataFrame(short).round(1)
lid_short_df.index.name = "language"
lid_short_df

In [ ]:
cm = pd.DataFrame(0.0, index=[name(l) for l in EVAL_LANGS], columns=EVAL_LANGS)
for lang in EVAL_LANGS:
    for code, share in LID_PRED[lang].value_counts(normalize=True).items():
        cm.loc[name(lang), code] = 100 * share

fig, ax = plt.subplots(figsize=(7, 0.5 * len(EVAL_LANGS) + 1.5))
ax.imshow(cm.values, cmap="Blues", vmin=0, vmax=100)
ax.set_xticks(range(len(EVAL_LANGS)), [f"{c}\n{name(c)}" for c in EVAL_LANGS], fontsize=8)
ax.set_yticks(range(len(cm)), cm.index)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        v = cm.values[i, j]
        if v >= 0.5:
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8, color="white" if v > 60 else "black")
ax.set_xlabel("predicted language"); ax.set_ylabel("true language (Belebele questions)")
ax.set_title("Stage 1: language ID on Belebele questions (% of each row)")
save_fig(fig, "1_langid_confusion.png")

## 5. Token tax — how many tokens does each language cost?

A language model does not read words. It reads **tokens**, pieces made by its tokenizer. Tokenizers learn their pieces mostly from English text. So an Indian-language word is often cut into many small pieces.

We measure this in two ways:

- **Fertility** = tokens ÷ words, on IndicCorp sentences. English is near 1.3. A value of 5 means an average word becomes 5 tokens.
- **Token tax** = tokens for a Belebele passage ÷ tokens for the same passage in English. Belebele is parallel, so both texts carry the same meaning. A token tax of 3.0 means the model must read 3 times more tokens for the same content.

More tokens means a longer input, more compute, and often lower accuracy. Section 14 checks if the token tax goes with the accuracy drop.

In [ ]:
TOKENIZERS = {
    "LLM": AutoTokenizer.from_pretrained(LLM_NAME),
    "translator (NLLB)": AutoTokenizer.from_pretrained(MT_NAME, revision=MT_REVISION),
    "retriever (e5)": AutoTokenizer.from_pretrained(EMB_NAME),
}

def n_tokens(tok, texts):
    # verbose=False: we only count tokens, so the "longer than the maximum length" notice does not apply
    return np.array([len(ids) for ids in tok(list(texts), add_special_tokens=False, verbose=False)["input_ids"]])

fert = pd.DataFrame({tname: {name(INDICCORP_LANGS[code]): n_tokens(tok, corpus[code][:1000]).sum()
                                                  / sum(len(l.split()) for l in corpus[code][:1000])
                             for code in INDICCORP_LANGS}
                     for tname, tok in TOKENIZERS.items()}).round(2)
fert.index.name = "language (IndicCorp)"

english_tokens = {tname: n_tokens(tok, KB[BASELINE]).sum() for tname, tok in TOKENIZERS.items()}
token_tax = pd.DataFrame({tname: {name(lang): n_tokens(tok, KB[lang]).sum() / english_tokens[tname] for lang in EVAL_LANGS}
                          for tname, tok in TOKENIZERS.items()}).round(2)
token_tax.index.name = "language (Belebele)"
display(fert)
display(token_tax)

fig, ax = plt.subplots(figsize=(7, 3.2))
order = token_tax["LLM"].sort_values()
ax.barh(order.index, order.values, color=["#1f77b4" if l == name(BASELINE) else "#9aa5b1" for l in order.index])
for i, v in enumerate(order.values):
    ax.text(v + 0.03, i, f"{v:.2f}×", va="center", fontsize=9)
ax.axvline(1, color="black", lw=0.8)
ax.set_xlim(0, order.max() * 1.15)                        # room for the value labels
ax.set_xlabel("LLM tokens for the same passages, relative to English")
ax.set_title("Token tax: the same content costs more tokens in Indian languages")
save_fig(fig, "2_token_tax.png")
del TOKENIZERS; free_memory()                             # ~1 GB of RAM that later cells do not need

## 6. Stage 2 — Retrieval: find the right passage

**Job.** The user asks a question. The bot must find the passage that holds the answer. Here the knowledge base is the 488 Belebele passages in the user's language, and the right answer is the passage that the question was written for.

**Method — dense embeddings.** An encoder turns every passage, and then the question, into one vector. The passage whose vector points most in the same direction as the question's vector (highest cosine similarity) ranks first. Dense vectors match meaning, not exact word forms. This matters for Indian languages: Tamil, Telugu and Marathi add many endings to one word root, so the word in the question is often not the exact word in the passage.

**Model — multilingual-e5-large-instruct** (Microsoft). A 24-layer XLM-RoBERTa encoder that turns a text into one vector of 1024 numbers: we average the output vectors of all tokens (mean pooling) and normalize the result. It was trained on question–passage pairs in about 100 languages, so that a question and its matching passage get vectors that point the same way. It is *instruction-tuned*: each question gets a one-line task description in front (`EMB_QUERY_PREFIX`); the passages get nothing. `sentence-transformers` handles the batching, the pooling and the normalization.

**Metric. Recall@1** = the share of questions whose right passage is ranked first. Recall@5 = right passage in the top 5.

In [ ]:
def load_retriever():
    model = SentenceTransformer(EMB_NAME, device=DEVICE, model_kwargs={"dtype": DTYPE})
    model.max_seq_length = 512                            # Belebele passages are shorter than this
    return model
LOADERS["retriever"] = load_retriever

def embed(texts, prefix=""):
    return gpu("retriever").encode([prefix + t for t in texts], batch_size=EMB_BATCH, normalize_embeddings=True,
                                   convert_to_tensor=True).float().cpu().numpy()

PASSAGE_VECS = {}
def retrieve(kb_lang, queries, k=5):
    # Return the ids of the k best passages from KB[kb_lang], best first.
    if kb_lang not in PASSAGE_VECS:
        PASSAGE_VECS[kb_lang] = embed(KB[kb_lang].tolist())
    scores = embed(list(queries), EMB_QUERY_PREFIX) @ PASSAGE_VECS[kb_lang].T     # cosine similarity
    return KB[kb_lang].index.to_numpy()[np.argsort(-scores, axis=1)[:, :k]]

In [ ]:
RET_TOP, rows = {}, []
for lang in EVAL_LANGS:
    gold = B[lang]["pid"].to_numpy()
    top = cached(f"retrieval_{slug(EMB_NAME)}_{lang}_{QTAG}", lambda: retrieve(lang, B[lang]["question"]))
    RET_TOP[lang] = top
    rows.append({"language": name(lang),
                 "Recall@1": 100 * (top[:, 0] == gold).mean(),
                 "Recall@5": 100 * (top == gold[:, None]).any(axis=1).mean()})
ret_table = pd.DataFrame(rows).set_index("language").round(1)
display(ret_table)

fig, ax = plt.subplots(figsize=(8, 3.2))
x = np.arange(len(EVAL_LANGS))
for i, (col, color) in enumerate([("Recall@1", "#1f77b4"), ("Recall@5", "#9aa5b1")]):
    vals = ret_table[col].to_numpy()
    ax.bar(x + (i - 0.5) * 0.38, vals, width=0.38, color=color, label=col)
    for xi, v in zip(x, vals):
        ax.text(xi + (i - 0.5) * 0.38, v + 1, f"{v:.0f}", ha="center", fontsize=8)
ax.set_xticks(x, ret_table.index, rotation=20, ha="right")
ax.set_ylim(0, 105); ax.set_ylabel("%")
ax.set_title("Stage 2: is the right passage ranked first?")
ax.legend(frameon=False, loc="lower left")
save_fig(fig, "3_retrieval.png")

## 7. Stage 3 — The answering model

**Model.** Qwen3-4B-Instruct (Alibaba), a decoder-only transformer (the same family as GPT). It reads the prompt from left to right and predicts the next token.

**4-bit weights.** 4 billion weights in bfloat16 take 8 GB, the whole memory of a laptop GPU. With `LLM_4BIT = True`, `bitsandbytes` stores each weight in 4 bits (the NF4 format) and turns it back to bfloat16 just before each multiplication. The model then needs 2.6 GB and loses little accuracy. A bigger model in 4 bits usually beats a smaller model in full precision: on 150 test questions, this model in 4 bits scored about 20 points higher in every language than Qwen3-1.7B in full precision.

**Prompt.** The instruction is in English for every language. Only the passage, the question and the options change language. Thus the comparison changes one thing only.

**How we read its answer: option scoring.** We do not let the model write freely and then parse the text. We run the prompt once and look at the probability the model gives to the next token being `A`, `B`, `C` or `D`. We keep these 4 numbers, apply softmax over them, and pick the highest. This is the method from the Belebele paper. It has three benefits:

- one forward pass per question (fast),
- no parsing errors (the answer is always one of 4 options),
- a confidence value for free (the winning probability).

**A detail for batches.** Prompts have different lengths, so we pad them. We pad on the **left**. Then the last position of every row is the real last token, and `[:, -1]` gives the next-token scores for all rows at once. We pass `logits_to_keep=1`, so the model applies its output layer to that last position only. The full output would be batch × length × 151,000 numbers, which wastes gigabytes of memory.

In [ ]:
llm_tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm_tok.padding_side = "left"
if llm_tok.pad_token is None:
    llm_tok.pad_token = llm_tok.eos_token
def load_llm():
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE) if LLM_4BIT else None
    return AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=DTYPE, quantization_config=quant,
                                                device_map={"": DEVICE}).eval()
LOADERS["llm"] = load_llm
LLM_TAG = slug(LLM_NAME) + ("-4bit" if LLM_4BIT else "")       # part of the cache file names
print(f"LLM weights: {gpu('llm').get_memory_footprint() / 1e9:.1f} GB")

LETTERS = ["A", "B", "C", "D"]
USE_CHAT = llm_tok.chat_template is not None                  # instruct models have a chat template
SYSTEM = "You are a helpful assistant for citizens. Answer only from the given passage."

def wrap(user_text):
    if USE_CHAT:
        messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_text}]
        # enable_thinking=False: Qwen3 models that can "think" answer at once (other templates ignore it)
        return llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    return user_text + "\nAnswer:"

def build_prompt(passage, question, options):
    lines = [f"Passage:\n{passage}", f"Question:\n{question}",
             "\n".join(f"{letter}. {opt}" for letter, opt in zip(LETTERS, options)),
             "Reply with only the letter of the correct option: A, B, C or D."]
    return wrap("\n\n".join(lines))

# Token id of each answer letter. Base models (no chat template) put a space before the letter.
LABEL_IDS = [llm_tok.encode(l if USE_CHAT else " " + l, add_special_tokens=False)[0] for l in LETTERS]
assert len(set(LABEL_IDS)) == 4, "the 4 letters must be 4 different tokens"

@torch.no_grad()
def score_prompts(prompts, desc=""):
    # Return (probabilities over A-D for each prompt, prompt length in tokens).
    probs = np.zeros((len(prompts), 4), dtype=np.float32)
    lengths = np.zeros(len(prompts), dtype=int)
    order = np.argsort([-len(p) for p in prompts])            # longest first: an out-of-memory error shows at once
    llm = gpu("llm")
    for batch_idx in tqdm(list(chunks(order, LLM_BATCH)), desc=desc, leave=False):
        enc = llm_tok([prompts[i] for i in batch_idx], return_tensors="pt", padding=True,
                      add_special_tokens=not USE_CHAT).to(DEVICE)
        positions = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)        # real positions despite left padding
        logits = llm(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=positions,
                     use_cache=False, logits_to_keep=1).logits[:, -1].float()   # output layer on the last position only
        probs[batch_idx] = torch.softmax(logits[:, LABEL_IDS], dim=-1).cpu().numpy()
        lengths[batch_idx] = enc["attention_mask"].sum(dim=1).cpu().numpy()
    return probs, lengths

RESULTS = {}     # (setting, lang) -> one row per question: pred, conf, p_gold, n_tokens, correct

def run_llm(setting, lang, passages, questions, options, inputs_from=""):
    # inputs_from names the models that made the inputs (translator, retriever), so the cache file changes with them
    def compute():
        prompts = [build_prompt(p, q, o) for p, q, o in zip(passages, questions, options)]
        return score_prompts(prompts, desc=f"{setting} | {name(lang)}")
    probs, lengths = cached(f"llm_{LLM_TAG}_{setting}_{inputs_from}_{lang}_{QTAG}", compute)
    gold = B[lang]["answer"].to_numpy()
    res = pd.DataFrame({"pred": probs.argmax(1), "conf": probs.max(1),
                        "p_gold": probs[np.arange(len(gold)), gold], "n_tokens": lengths}, index=B[lang].index)
    res["correct"] = res["pred"].to_numpy() == gold
    RESULTS[(setting, lang)] = res
    return res

In [ ]:
# Sanity check on one English question: see the exact prompt and the 4 probabilities
b = B[BASELINE].iloc[0]
example_prompt = build_prompt(b["flores_passage"], b["question"], b["options"])
print(example_prompt)
p, n = score_prompts([example_prompt])
print("probabilities:", dict(zip(LETTERS, p[0].round(3))), "| correct:", LETTERS[b["answer"]], "| prompt tokens:", n[0])

## 8. Stage 3A — Direct route, with the correct passage

The model gets the correct passage, the question and the options, all in the test language. This isolates the **reading** step: retrieval cannot fail here, because we give the right passage.

Chance level is 25% (4 options).

In [ ]:
for lang in EVAL_LANGS:
    b = B[lang]
    run_llm("direct_gold", lang, b["flores_passage"], b["question"], b["options"])

pd.DataFrame({name(lang): {"accuracy (%)": 100 * RESULTS[("direct_gold", lang)]["correct"].mean(),
                           "avg prompt tokens": RESULTS[("direct_gold", lang)]["n_tokens"].mean()}
              for lang in EVAL_LANGS}).T.round(1)

## 9. Translation to English with NLLB

**Model.** NLLB-200 distilled 1.3B (Meta). It is an encoder-decoder transformer, the same design as the original *Attention Is All You Need* model. The encoder reads the source sentence. The decoder writes the English sentence one token at a time. A language tag tells it the source language, and the first output token (`eng_Latn`) tells it the target language.

**Sentence by sentence.** NLLB was trained on single sentences. Long inputs can make it skip text. So we cut each passage at `.`, `?`, `!` and `।`, translate the sentences, and join them again.

**Translation quality: chrF.** Belebele gives us the real English passage, so we can score the machine translation against it. chrF compares character n-grams of the two texts (like an F1 score over character pieces). It is 0–100; higher is better. chrF works better than word overlap for languages with long words.

**Beam search.** With `MT_BEAMS = 4` the decoder keeps the 4 best partial translations at every step instead of only the best one (greedy). This costs time but gives better translations.

All translations are saved in a cache file, so each sentence is translated only once.

In [ ]:
LOADERS["translator"] = lambda: AutoModelForSeq2SeqLM.from_pretrained(
    MT_NAME, revision=MT_REVISION, dtype=MT_DTYPE, device_map={"": DEVICE}).eval()
MT_TOKS = {}
def mt_tokenizer(src):
    # One tokenizer per source language: it puts the source-language tag in front of every input.
    if src not in MT_TOKS:
        MT_TOKS[src] = AutoTokenizer.from_pretrained(MT_NAME, revision=MT_REVISION, src_lang=src)
    return MT_TOKS[src]

MT_TAG = f"{slug(MT_NAME)}_beams{MT_BEAMS}"                 # part of the cache file names
TCACHE_PATH = CACHE_DIR / f"translations_{MT_TAG}.json"
TCACHE = json.loads(TCACHE_PATH.read_text(encoding="utf-8")) if (USE_CACHE and TCACHE_PATH.exists()) else {}

@torch.no_grad()
def translate(texts, src, tgt="eng_Latn"):
    texts = list(texts)
    if src == tgt:
        return texts
    key = lambda t: f"{src}>{tgt}|{t}"
    todo = sorted({t for t in texts if key(t) not in TCACHE}, key=len, reverse=True)
    if todo:
        mt_tok = mt_tokenizer(src)                              # adds the source-language tag
        tgt_id = mt_tok.convert_tokens_to_ids(tgt)              # forced first output token = target language
        mt_model = gpu("translator")
        for batch in tqdm(list(chunks(todo, MT_BATCH)), desc=f"translate {src} -> {tgt}", leave=False):
            enc = mt_tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=MT_MAX_TOKENS).to(DEVICE)
            # max_length, not max_new_tokens: NLLB's own config already sets max_length, and setting both warns
            out = mt_model.generate(**enc, forced_bos_token_id=tgt_id, num_beams=MT_BEAMS, max_length=MT_MAX_TOKENS)
            for t, o in zip(batch, mt_tok.batch_decode(out, skip_special_tokens=True)):
                TCACHE[key(t)] = o
        TCACHE_PATH.write_text(json.dumps(TCACHE, ensure_ascii=False), encoding="utf-8")
    return [TCACHE[key(t)] for t in texts]

SENTENCE_END = re.compile(r"(?<=[.!?।॥])\s+")

def translate_passages(passages, src, tgt="eng_Latn"):
    # Translate long texts sentence by sentence, then join the sentences again.
    if src == tgt:
        return list(passages)
    split = [[s for s in SENTENCE_END.split(p.strip()) if s] for p in passages]
    flat = translate([s for sents in split for s in sents], src, tgt)
    out, i = [], 0
    for sents in split:
        out.append(" ".join(flat[i:i + len(sents)]))
        i += len(sents)
    return out

def translate_by_source(texts, sources, fn=translate):
    # Translate each text from its own source language (a Belebele code) to English.
    texts, sources = list(texts), list(sources)
    out = list(texts)
    for src in set(sources):
        idx = [i for i, s in enumerate(sources) if s == src]
        for i, t in zip(idx, fn([texts[i] for i in idx], src)):
            out[i] = t
    return out

In [ ]:
PIVOT, chrf_rows = {}, []
first = ~B[BASELINE]["pid"].duplicated().to_numpy()          # one row per passage, for corpus-level chrF
for lang in EVAL_LANGS:
    if lang == BASELINE:
        continue
    b, src = B[lang], lang
    passages_en = translate_passages(b["flores_passage"], src)
    questions_en = translate(b["question"], src)
    flat_options = translate([o for opts in b["options"] for o in opts], src)
    PIVOT[lang] = pd.DataFrame({"passage": passages_en, "question": questions_en,
                                "options": [flat_options[i:i + 4] for i in range(0, len(flat_options), 4)]},
                               index=b.index)
    ref_passages = B[BASELINE]["flores_passage"].tolist()
    PIVOT[lang]["chrf_passage"] = [sacrebleu.sentence_chrf(h, [r]).score for h, r in zip(passages_en, ref_passages)]
    chrf_rows.append({
        "language": name(lang),
        "chrF passage": sacrebleu.corpus_chrf(list(np.array(passages_en)[first]), [list(np.array(ref_passages)[first])]).score,
        "chrF question": sacrebleu.corpus_chrf(questions_en, [B[BASELINE]["question"].tolist()]).score,
    })
chrf_df = pd.DataFrame(chrf_rows).set_index("language").round(1)
display(chrf_df)

# One example: the machine translation next to the real English text
show = "tam_Taml" if "tam_Taml" in PIVOT else next(iter(PIVOT))
q = qids[0]
print(f"{name(show)} question  : {B[show].loc[q, 'question']}")
print(f"NLLB -> English     : {PIVOT[show].loc[q, 'question']}")
print(f"real English        : {B[BASELINE].loc[q, 'question']}")

## 10. Stage 3B — Pivot route, with the correct passage

Same model, same prompt, same correct passage, but the passage, question and options are now NLLB's English translation. English needs no translation, so its pivot score equals its direct score.

**Compare with 3A.** If the pivot route scores higher, the LLM reads English better than it reads the original language, and translation is worth its cost. If it scores lower, the translation loses information.

In [ ]:
for lang in EVAL_LANGS:
    if lang == BASELINE:
        RESULTS[("pivot_gold", lang)] = RESULTS[("direct_gold", lang)]
        continue
    p = PIVOT[lang]
    run_llm("pivot_gold", lang, p["passage"], p["question"], p["options"], inputs_from=MT_TAG)

pd.DataFrame({name(lang): {"direct (%)": 100 * RESULTS[("direct_gold", lang)]["correct"].mean(),
                           "pivot (%)": 100 * RESULTS[("pivot_gold", lang)]["correct"].mean()}
              for lang in EVAL_LANGS}).T.round(1)

## 11. End to end — every stage for real

Sections 4–10 tested each stage alone, each time with correct inputs. A real chatbot chains the stages, so one error passes to the next stage. Here every question goes through the full pipeline:

1. **Language ID** predicts the language of the question.
2. **Retrieval** searches the knowledge base *of the predicted language* (English if we have no knowledge base in that language) and takes the top passage.
3. **Answer.**
   - *Direct*: the LLM reads the retrieved passage and the original question.
   - *Pivot*: NLLB translates the passage (from its knowledge-base language) and the question and options (from the predicted language) to English. Then the LLM answers.

A wrong language label now has a cost: the bot searches the wrong knowledge base and gives the translator the wrong source tag.

The cell runs one stage for all languages before it starts the next stage. On a small GPU the models take turns, so this order keeps the switches few.

In [ ]:
E2E_INFO, E2E_INPUTS = {}, {}
for lang in EVAL_LANGS:     # 1. language ID, then retrieval in the knowledge base of the detected language
    b, detected = B[lang], LID_PRED[lang]
    top_pid = pd.Series(0, index=b.index)
    for kl, idx in detected.groupby(detected).groups.items():
        top_pid.loc[idx] = retrieve(kl, b.loc[idx, "question"], k=1)[:, 0]
    E2E_INFO[lang] = pd.DataFrame({"detected": detected, "top_pid": top_pid,
                                   "lid_ok": detected == lang, "retrieval_ok": top_pid == b["pid"]})

for lang in EVAL_LANGS:     # 2. pivot route: translate the retrieved passage, the question and the options
    b, info = B[lang], E2E_INFO[lang]
    passages = [KB[kl].loc[pid] for kl, pid in zip(info["detected"], info["top_pid"])]
    passages_en = translate_by_source(passages, info["detected"], fn=translate_passages)
    questions_en = translate_by_source(b["question"], info["detected"])
    flat_sources = [s for s in info["detected"] for _ in range(4)]
    flat_options = translate_by_source([o for opts in b["options"] for o in opts], flat_sources)
    options_en = [flat_options[i:i + 4] for i in range(0, len(flat_options), 4)]
    E2E_INPUTS[lang] = passages, passages_en, questions_en, options_en

for lang in EVAL_LANGS:     # 3. the LLM answers, on both routes
    b = B[lang]
    passages, passages_en, questions_en, options_en = E2E_INPUTS[lang]
    run_llm("direct_e2e", lang, passages, b["question"], b["options"], inputs_from=slug(EMB_NAME))
    run_llm("pivot_e2e", lang, passages_en, questions_en, options_en, inputs_from=f"{slug(EMB_NAME)}_{MT_TAG}")

DIRECT_E2E, PIVOT_E2E = "direct_e2e", "pivot_e2e"
pd.DataFrame({name(lang): {"language ID right (%)": 100 * E2E_INFO[lang]["lid_ok"].mean(),
                           "right passage (%)": 100 * E2E_INFO[lang]["retrieval_ok"].mean(),
                           "direct e2e (%)": 100 * RESULTS[(DIRECT_E2E, lang)]["correct"].mean(),
                           "pivot e2e (%)": 100 * RESULTS[(PIVOT_E2E, lang)]["correct"].mean()}
              for lang in EVAL_LANGS}).T.round(1)

## 12. Comparison: where does quality drop?

### 12.1 The summary table

One row per language, one column per stage. **Gap** = this language minus English, in percentage points.

**Confidence intervals.** With 900 questions, a score of 60% could be 57% or 63% on another set of questions. The 95% interval (Wilson method, from `statsmodels`) shows this range. Two languages whose intervals overlap a lot may not really differ.

In [ ]:
def acc(setting, lang):
    return 100 * RESULTS[(setting, lang)]["correct"].mean()

def wilson(correct):
    # 95% confidence interval for an accuracy, in %
    low, high = proportion_confint(int(np.sum(correct)), len(correct), alpha=0.05, method="wilson")
    return 100 * low, 100 * high

summary = pd.DataFrame([{
    "language": name(lang),
    "1 LangID (%)": lid_acc[lang],
    "2 R@1 (%)": 100 * (RET_TOP[lang][:, 0] == B[lang]["pid"].to_numpy()).mean(),
    "chrF to EN": chrf_df["chrF passage"].get(name(lang), np.nan),
    "3A direct (%)": acc("direct_gold", lang),
    "3B pivot (%)": acc("pivot_gold", lang),
    "E2E direct (%)": acc(DIRECT_E2E, lang),
    "E2E pivot (%)": acc(PIVOT_E2E, lang),
    "gap 3A vs EN": acc("direct_gold", lang) - acc("direct_gold", BASELINE),
    "gap E2E vs EN": acc(DIRECT_E2E, lang) - acc(DIRECT_E2E, BASELINE),
    "token tax": token_tax.loc[name(lang), "LLM"],
} for lang in EVAL_LANGS]).set_index("language").round(1)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
x = np.arange(len(EVAL_LANGS))
for i, (setting, label, color) in enumerate([("direct_gold", "3A direct", "#1f77b4"), ("pivot_gold", "3B pivot via English", "#9aa5b1")]):
    vals = [acc(setting, l) for l in EVAL_LANGS]
    ci = np.array([wilson(RESULTS[(setting, l)]["correct"].to_numpy()) for l in EVAL_LANGS])
    err = np.vstack([np.array(vals) - ci[:, 0], ci[:, 1] - np.array(vals)])
    ax.bar(x + (i - 0.5) * 0.38, vals, 0.38, yerr=err, capsize=3, color=color, label=label)
ax.axhline(25, color="grey", ls=":", lw=1)
ax.text(1.01, 25, "chance\n25%", transform=ax.get_yaxis_transform(), va="center", fontsize=8, color="grey")
ax.axhline(acc("direct_gold", BASELINE), color="black", ls="--", lw=0.8)
ax.set_xticks(x, [name(l) for l in EVAL_LANGS], rotation=20, ha="right")
ax.set_ylim(0, 100); ax.set_ylabel("accuracy (%)")
ax.set_title("Answer accuracy with the correct passage (bars: 95% interval; dashed: English)")
ax.legend(frameon=False, loc="upper right")
save_fig(fig, "4_answer_accuracy.png")

### 12.2 Paired comparison: which questions flip?

Every question exists in English and in the test language. So we can count, question by question:

- **both right**, **both wrong**: the language changed nothing.
- **only English right**: a **drop** caused by the language.
- **only this language right**: the language helped (usually luck).

**McNemar test** (exact version, from `statsmodels`). If language had no effect, drops and gains would be equally likely, like coin flips. The test asks: how surprising is the split we see between drops and gains? A small *p* (below 0.05) means the drop is real, not noise. Only the questions that flipped matter for this test.

In [ ]:
def paired(res_a, res_b):
    a, b = res_a["correct"], res_b["correct"]
    table = [[int((a & b).sum()), int((a & ~b).sum())],      # rows: first right / wrong
             [int((~a & b).sum()), int((~a & ~b).sum())]]    # columns: second right / wrong
    return {"both right": table[0][0], "both wrong": table[1][1],
            "only first right": table[0][1], "only second right": table[1][0],
            "McNemar p": mcnemar(table, exact=True).pvalue}

eng = RESULTS[("direct_gold", BASELINE)]
flips = pd.DataFrame({name(l): paired(eng, RESULTS[("direct_gold", l)]) for l in EVAL_LANGS if l != BASELINE}).T
flips = flips.rename(columns={"only first right": "only English right (drop)", "only second right": "only this language right"})
pivot_vs_direct = pd.DataFrame({name(l): paired(RESULTS[("pivot_gold", l)], RESULTS[("direct_gold", l)])
                                for l in EVAL_LANGS if l != BASELINE}).T
pivot_vs_direct = pivot_vs_direct.rename(columns={"only first right": "fixed by translation", "only second right": "broken by translation"})
for table in (flips, pivot_vs_direct):
    counts = [c for c in table.columns if c != "McNemar p"]
    table[counts] = table[counts].astype(int)
    table["McNemar p"] = table["McNemar p"].astype(float).map(lambda v: f"{v:.2g}")
print("English vs each language (3A direct, correct passage):"); display(flips)
print("Pivot vs direct, same language (does translation help?):"); display(pivot_vs_direct)

### 12.3 The drop map

Each cell is *this language minus English* in percentage points, for one stage. Red = a loss. Read across a row to find the stage where a language loses most.

In [ ]:
stage_cols = ["1 LangID (%)", "2 R@1 (%)", "3A direct (%)", "3B pivot (%)", "E2E direct (%)", "E2E pivot (%)"]
drop_map = summary[stage_cols].sub(summary.loc[name(BASELINE), stage_cols], axis=1).drop(index=name(BASELINE))

fig, ax = plt.subplots(figsize=(9, 0.55 * len(drop_map) + 1.6))
lim = float(np.clip(np.nanpercentile(np.abs(drop_map.values), 95), 10, 100))   # scale from typical drops, not one outlier
ax.imshow(drop_map.values, cmap="RdBu", vmin=-lim, vmax=lim, aspect="auto")
ax.set_xticks(range(len(stage_cols)), [c.replace(" (%)", "") for c in stage_cols])
ax.set_yticks(range(len(drop_map)), drop_map.index)
for i in range(drop_map.shape[0]):
    for j in range(drop_map.shape[1]):
        v = drop_map.values[i, j]
        label = "0" if abs(v) < 0.5 else f"{v:+.0f}"          # no "-0"
        ax.text(j, i, label, ha="center", va="center", fontsize=9, color="white" if abs(v) > 0.6 * lim else "black")
ax.set_title("Drop map: points lost against English at each stage")
save_fig(fig, "5_drop_map.png")

### 12.4 Where each language loses its questions (end to end, direct route)

Every wrong end-to-end answer gets **one** cause, checked in pipeline order:

1. **language ID** — the predicted language was wrong;
2. **retrieval** — else, the top passage was not the right one;
3. **reading** — else, the model had the right passage and still chose wrongly.

In [ ]:
def e2e_cause(lang, setting):
    res, info = RESULTS[(setting, lang)], E2E_INFO[lang]
    cause = pd.Series("correct", index=res.index)
    wrong = ~res["correct"]
    cause[wrong & ~info["lid_ok"]] = "lost at language ID"
    cause[wrong & info["lid_ok"] & ~info["retrieval_ok"]] = "lost at retrieval"
    cause[wrong & info["lid_ok"] & info["retrieval_ok"]] = "lost at reading"
    return cause

CAUSES = ["correct", "lost at language ID", "lost at retrieval", "lost at reading"]
loss = pd.DataFrame({name(l): e2e_cause(l, DIRECT_E2E).value_counts(normalize=True).reindex(CAUSES, fill_value=0) * 100
                     for l in EVAL_LANGS}).T.round(1)
display(loss)

fig, ax = plt.subplots(figsize=(9, 0.5 * len(loss) + 1.4))
colors = {"correct": "#1f77b4", "lost at language ID": "#d62728", "lost at retrieval": "#ff7f0e", "lost at reading": "#bdbdbd"}
left = np.zeros(len(loss))
for c in CAUSES:
    ax.barh(loss.index, loss[c], left=left, color=colors[c], label=c)
    for i, (l, v) in enumerate(zip(left, loss[c])):
        if v >= 6:
            ax.text(l + v / 2, i, f"{v:.0f}", ha="center", va="center", fontsize=8, color="white" if c != "lost at reading" else "black")
    left += loss[c].to_numpy()
ax.invert_yaxis(); ax.set_xlim(0, 100)
ax.set_title("End to end (direct route): where each language loses its questions (% of questions)", pad=28)
ax.legend(ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.0), loc="lower center", fontsize=8)
save_fig(fig, "6_e2e_losses.png")

## 13. Failure examples

We look at the **drops** of Section 12.2: questions that English answers correctly but this language does not (direct route, correct passage). Each drop gets a category:

| Category | Rule |
|---|---|
| fixed by translation | the pivot route answers it correctly |
| translation damaged the passage | the pivot route is also wrong, and the passage chrF is below `CHRF_DAMAGED` |
| hard even after translation | the pivot route is also wrong, and the translation looks fine |

We also mark questions whose correct option contains a **number**. Numbers can be written in native digits (for example `४३६` or `৪৩৬`), and tokenizers often cut them badly.

The English text in brackets is the real Belebele English version of the same option. It lets you read every failure without knowing the language.

In [ ]:
def has_number(text):
    return any(ch.isdigit() for ch in text)          # also true for Devanagari, Bengali, Tamil digits

def drop_category(lang, qid):
    if RESULTS[("pivot_gold", lang)].loc[qid, "correct"]:
        return "fixed by translation"
    if PIVOT[lang].loc[qid, "chrf_passage"] < CHRF_DAMAGED:
        return "translation damaged the passage"
    return "hard even after translation"

rng = np.random.default_rng(SEED)
eng = RESULTS[("direct_gold", BASELINE)]
category_counts, examples = {}, []
for lang in EVAL_LANGS:
    if lang == BASELINE:
        continue
    d = RESULTS[("direct_gold", lang)]
    drops = d.index[eng["correct"].to_numpy() & ~d["correct"].to_numpy()]
    cats = pd.Series([drop_category(lang, q) for q in drops], index=drops, dtype=object)
    category_counts[name(lang)] = cats.value_counts().to_dict() | {"total drops": len(drops)}
    picks = rng.choice(drops, size=min(N_FAILURE_EXAMPLES, len(drops)), replace=False) if len(drops) else []
    for q in picks:
        b, e = B[lang].loc[q], B[BASELINE].loc[q]
        pred, gold = int(d.loc[q, "pred"]), int(b["answer"])
        examples.append({
            "language": name(lang), "qid": int(q), "category": cats[q],
            "number in answer": has_number(b["options"][gold]),
            "question": b["question"], "English original": e["question"],
            "NLLB translation": PIVOT[lang].loc[q, "question"],
            "correct option": f"{LETTERS[gold]}. {b['options'][gold]}  [{e['options'][gold]}]",
            "model chose": f"{LETTERS[pred]}. {b['options'][pred]}  [{e['options'][pred]}]",
            "confidence": round(float(d.loc[q, "conf"]), 2),
            "passage chrF": round(float(PIVOT[lang].loc[q, "chrf_passage"]), 1),
        })

column_order = ["fixed by translation", "translation damaged the passage", "hard even after translation", "total drops"]
drop_categories = pd.DataFrame(category_counts).T.reindex(columns=column_order).fillna(0).astype(int)
drop_categories.index.name = "language"
failures = pd.DataFrame(examples)
display(drop_categories)
failures

## 14. Two diagnostics

**Token tax against the drop.** Does a language lose more accuracy when the LLM needs more tokens for it? With only 5 languages besides English this is a hint, not a proof: other things also differ between languages, such as how much training text the model saw.

**Number questions.** Accuracy on questions whose correct option contains a number, against all other questions.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
others = [l for l in EVAL_LANGS if l != BASELINE]
xs = [token_tax.loc[name(l), "LLM"] for l in others]
ys = [summary.loc[name(l), "gap 3A vs EN"] for l in others]
ax.scatter(xs, ys, color="#1f77b4")
for x_, y_, l in zip(xs, ys, others):
    ax.annotate(name(l), (x_, y_), textcoords="offset points", xytext=(5, 4), fontsize=8)
ax.axhline(0, color="black", lw=0.8)
ax.margins(x=0.15)                                        # keep edge labels inside the frame
ax.set_xlabel("token tax (LLM tokens relative to English)")
ax.set_ylabel("accuracy gap vs English (points)")
ax.set_title("More tokens for the same content, bigger drop?")
save_fig(fig, "7_token_tax_vs_gap.png")
if len(xs) > 2:
    print(f"Pearson r: {np.corrcoef(xs, ys)[0, 1]:.2f}")

num_mask = {l: B[l].apply(lambda r: has_number(r["options"][r["answer"]]), axis=1) for l in EVAL_LANGS}
numbers = pd.DataFrame({name(l): {
    "number questions": int(num_mask[BASELINE].sum()),
    "accuracy, number questions (%)": 100 * RESULTS[("direct_gold", l)]["correct"][num_mask[BASELINE].to_numpy()].mean(),
    "accuracy, other questions (%)": 100 * RESULTS[("direct_gold", l)]["correct"][~num_mask[BASELINE].to_numpy()].mean(),
} for l in EVAL_LANGS}).T.round(1)
numbers["number questions"] = numbers["number questions"].astype(int)
numbers

## 15. Write the report

This cell writes the deliverable to `bhashacore_outputs/`:

- `report.md` — the comparison report: tables, charts, failure categories and examples;
- `summary.csv` — the summary table;
- `per_question.csv` — every prediction (language, question, setting, chosen option, right or wrong);
- `failures.csv` — the failure examples;
- `figures/*.png` — all charts.

Open `report.md` in VS Code (Ctrl+Shift+V) or on GitHub to see it with the charts.

In [ ]:
summary.to_csv(OUT_DIR / "summary.csv")
failures.to_csv(OUT_DIR / "failures.csv", index=False)
pd.concat([res.assign(language=lang, setting=setting) for (setting, lang), res in RESULTS.items()]) \
  .reset_index().to_csv(OUT_DIR / "per_question.csv", index=False)

worst = summary.drop(index=name(BASELINE))["gap 3A vs EN"].idxmin()
best_pivot = (summary["3B pivot (%)"] - summary["3A direct (%)"]).drop(index=name(BASELINE))
report = [
    "# BhashaCore comparison report", "",
    f"Generated {pd.Timestamp.now():%Y-%m-%d %H:%M} · {len(qids)} paired Belebele questions per language · "
    f"language ID `{LID_REPO}/{LID_FILE}` · retriever `{EMB_NAME}` · LLM `{LLM_NAME}`{' (4-bit)' if LLM_4BIT else ''} · "
    f"translator `{MT_NAME}` (beams {MT_BEAMS})", "",
    "## Headline", "",
    f"- English accuracy with the correct passage: **{acc('direct_gold', BASELINE):.1f}%** (chance = 25%).",
    f"- Largest drop: **{worst}**, {summary.loc[worst, 'gap 3A vs EN']:+.1f} points against English.",
    f"- Translation to English helps most for **{best_pivot.idxmax()}** ({best_pivot.max():+.1f} points) "
    f"and least for **{best_pivot.idxmin()}** ({best_pivot.min():+.1f} points).", "",
    "## 1. Summary by stage", "", md_table(summary), "",
    "Columns: language-ID accuracy on questions; Recall@1 of the retriever; chrF of NLLB's English against the real English passage; "
    "answer accuracy with the correct passage (3A direct, 3B pivot); end-to-end accuracy; gap = language minus English (points); "
    "token tax = LLM tokens relative to English.", "",
    "![answer accuracy](figures/4_answer_accuracy.png)", "",
    "## 2. Drop map", "", "![drop map](figures/5_drop_map.png)", "",
    "## 3. Where questions are lost end to end (direct route)", "", md_table(loss), "", "![losses](figures/6_e2e_losses.png)", "",
    "## 4. Paired flips against English (3A direct, correct passage)", "", md_table(flips), "",
    "## 5. Does translation help? (pivot vs direct, same language)", "", md_table(pivot_vs_direct), "",
    "## 6. Language identification", "", md_table(lid_short_df), "", "![confusion](figures/1_langid_confusion.png)", "",
    "## 7. Retrieval", "", "![retrieval](figures/3_retrieval.png)", "",
    "## 8. Token tax", "", md_table(token_tax, 2), "", "![token tax](figures/2_token_tax.png)", "",
    "![token tax vs gap](figures/7_token_tax_vs_gap.png)", "",
    "## 9. Number questions", "", md_table(numbers), "",
    "## 10. Why answers dropped (questions English got right)", "", md_table(drop_categories, 0), "",
    "## 11. Failure examples", "",
    "Text in [brackets] is the real English version of the same option.", "",
]
for lang_name, group in failures.groupby("language", sort=False):
    report += [f"### {lang_name}", ""]
    for _, r in group.iterrows():
        report += [f"- **qid {r['qid']}** · {r['category']}{' · number in answer' if r['number in answer'] else ''} · confidence {r['confidence']}",
                   f"  - Question: {r['question']}", f"  - English original: {r['English original']}",
                   f"  - NLLB translation: {r['NLLB translation']}",
                   f"  - Correct: {r['correct option']}", f"  - Model chose: {r['model chose']}", ""]
report += ["## Limits", "",
           "- Belebele passages are news text, not government FAQs; the method transfers, the numbers may not.",
           "- One LLM and one prompt; another model or a native-language instruction can change the ranking.",
           "- Failure categories come from simple rules. Read the examples before you trust a count.",
           "- Intervals are 95% Wilson intervals; McNemar p-values compare paired questions."]
(OUT_DIR / "report.md").write_text("\n".join(report), encoding="utf-8")
print("wrote", OUT_DIR / "report.md")

## 16. Chatbot demo — free-text answers

Until now the model chose A–D. A real chatbot writes an answer. This function runs the full pipeline on any message, with no options:

- **direct**: detect the language → search that language's knowledge base → the LLM answers in that language.
- **pivot**: detect the language → translate the question to English → search the English knowledge base → the LLM answers in English → translate the answer back.

On a small GPU each reply loads models a few times (5–10 seconds each), so a reply takes a little while. Try your own questions in the last cell. The knowledge base holds only the Belebele news passages, so ask about their topics.

In [ ]:
@torch.no_grad()
def generate(prompt_text, max_new_tokens=256):          # Indian scripts need ~4-7x more tokens than English
    enc = llm_tok(prompt_text, return_tensors="pt", add_special_tokens=not USE_CHAT).to(DEVICE)
    out = gpu("llm").generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], max_new_tokens=max_new_tokens,
                              do_sample=False, temperature=None, top_p=None, top_k=None, pad_token_id=llm_tok.pad_token_id)
    return llm_tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def chat_prompt(passage, question, reply_language):
    return wrap(f"Passage:\n{passage}\n\nQuestion: {question}\n\n"
                f"Answer in {reply_language}, in one or two sentences, using only the passage.")

def bhashacore_chat(message, route="direct"):
    lang = detect_language([message])[0]
    if route == "direct":
        pid = retrieve(lang, [message], k=1)[0, 0]
        answer = generate(chat_prompt(KB[lang].loc[pid], message, name(lang)))
    else:
        question_en = translate([message], lang)[0]
        pid = retrieve(BASELINE, [question_en], k=1)[0, 0]
        answer_en = generate(chat_prompt(KB[BASELINE].loc[pid], question_en, "English"))
        answer = translate([answer_en], BASELINE, lang)[0]
    return {"detected language": name(lang), "passage id": int(pid), "answer": answer}

In [ ]:
q = qids[1]
demo_rows = []
for lang in [l for l in ["hin_Deva", "tam_Taml", "tel_Telu"] if l in EVAL_LANGS]:
    message = B[lang].loc[q, "question"]
    for route in ("direct", "pivot"):
        out = bhashacore_chat(message, route)
        demo_rows.append({"language": name(lang), "route": route, "message": message, **out,
                          "right passage": out["passage id"] == B[lang].loc[q, "pid"]})
print("English question:", B[BASELINE].loc[q, "question"])
print("Correct option   :", B[BASELINE].loc[q, "options"][B[BASELINE].loc[q, "answer"]])
pd.DataFrame(demo_rows)

In [ ]:
# Your turn: type any question in any language
bhashacore_chat("बिजली बिल की शिकायत कहाँ करें?", route="pivot")

## 17. Next steps

- **Bigger LLM.** On a GPU with 20 GB or more, set `LLM_4BIT = False`, or try `Qwen/Qwen3-8B`. Does the gap shrink, or does every language rise together?
- **Better translator.** AI4Bharat's IndicTrans2 is made for Indian languages, but it needs an access request on Hugging Face. Does the pivot route improve for Tamil and Telugu?
- **More languages.** Add Gujarati, Kannada, Malayalam, Odia, Punjabi or Urdu to `EVAL_LANGS` and `INDICCORP_LANGS`. Does the token tax still predict the drop?
- **Native instructions.** Write the prompt instruction in Hindi for the Hindi run. Does the score change?
- **Government text.** Replace the Belebele passages with real government FAQ pages in several languages. Keep the same pipeline and metrics.